In [36]:
# Imports standards
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import json
import os
import time
from collections import Counter
# Imports scipy pour matrices sparse
from scipy.sparse import csr_matrix, coo_matrix
# Imports LightFM
try:
    from lightfm import LightFM
    from lightfm.data import Dataset
    from lightfm.evaluation import precision_at_k, recall_at_k, auc_score
    from lightfm.cross_validation import random_train_test_split
    print("✅ LightFM installé et disponible")
    LIGHTFM_AVAILABLE = True
except ImportError:
    print("❌ LightFM n'est pas installé!")
    print("   Installer avec: pip install lightfm")
    LIGHTFM_AVAILABLE = False
    raise ImportError("LightFM est requis pour ce notebook")
# Configuration des warnings et affichage
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
# Style des visualisations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
print("\n✅ Imports terminés")


✅ LightFM installé et disponible

✅ Imports terminés


In [52]:
def create_age_bins(customers_df):
    """
    Crée des tranches d'âge catégorisées pour les features utilisateur.

    Args:
        customers_df (pd.DataFrame): DataFrame contenant la colonne 'age'

    Returns:
        pd.DataFrame: DataFrame avec colonne 'age_bin' ajoutée
    """
    bins = [0, 18, 25, 35, 45, 55, 65, 100]
    labels = ['0-18', '19-25', '26-35', '36-45', '46-55', '56-65', '65+']

    customers_df['age_bin'] = pd.cut(customers_df['age'], bins=bins, labels=labels, right=False)
    customers_df['age_bin'] = customers_df['age_bin'].astype('category')
    customers_df['age_bin'] = customers_df['age_bin'].cat.add_categories('Inconnu').fillna('Inconnu')

    return customers_df


def build_interaction_matrix(data, user_id_map, item_id_map):
    """
    Construit une matrice d'interactions sparse CSR.

    Args:
        data (pd.DataFrame): DataFrame avec 'customer_id', 'article_id'
        user_id_map (dict): Mapping utilisateur → index
        item_id_map (dict): Mapping article → index

    Returns:
        tuple: (matrice sparse, n_users, n_items)
    """
    rows = data['customer_id'].map(user_id_map)
    cols = data['article_id'].map(item_id_map)
    values = np.ones(len(data))

    matrix = sparse.csr_matrix(
        (values, (rows, cols)),
        shape=(len(user_id_map), len(item_id_map))
    )

    return matrix


def build_interaction_matrix_with_dataset(dataset, data_df):
    """
    Construit une matrice d'interactions sparse en utilisant un objet Dataset
    de LightFM déjà "fitté".

    Cette fonction remplace la création manuelle de la matrice CSR.
    Elle attend les ID originaux (customer_id, article_id) car le dataset
    applique ses propres mappings internes.

    Args:
        dataset (lightfm.data.Dataset): L'objet Dataset qui a déjà été
                                        fitté avec les utilisateurs et articles.
        data_df (pd.DataFrame): DataFrame contenant les interactions
                                (doit avoir 'customer_id' et 'article_id').

    Returns:
        scipy.sparse.coo_matrix: La matrice d'interactions (LightFM renvoie du COO).
    """

    # 1. Préparer l'itérable (user_id, item_id)
    # Nous utilisons drop_duplicates pour garantir une seule interaction par paire
    # (modèle implicite).
    interactions_data = data_df[['customer_id', 'article_id']].drop_duplicates()
    interactions_iterable = zip(interactions_data['customer_id'],
                                interactions_data['article_id'])

    # 2. Construire la matrice
    # Renvoie un tuple (interactions, poids)
    (interactions_matrix, weights_matrix) = dataset.build_interactions(interactions_iterable)

    return interactions_matrix


print("🛍️  Chargement articles.csv...")
df_articles = pd.read_csv(f"./data/articles.csv")
print(f"   ✓ {len(df_articles):,} articles chargés")

print("\n👥 Chargement customers.csv...")
df_customers = pd.read_csv(f"./data/customers.csv")
print(f"   ✓ {len(df_customers):,} clients chargés")

print("\n📊 Chargement transactions_train.csv...")
df_transactions = pd.read_csv(f"./data/transactions_train.csv")
print(f"   ✓ {len(df_transactions):,} transactions chargées")

print("\n🎉 Tous les datasets H&M ont été chargés avec succès !")

print("\n" + "=" * 70)
print("STRATÉGIE COMBINE")
print("=" * 70)

🛍️  Chargement articles.csv...
   ✓ 105,542 articles chargés

👥 Chargement customers.csv...
   ✓ 1,371,980 clients chargés

📊 Chargement transactions_train.csv...
   ✓ 31,788,324 transactions chargées

🎉 Tous les datasets H&M ont été chargés avec succès !

STRATÉGIE COMBINE


In [53]:
# 4. Prétraitement des features utilisateur
print(f"\n🔧 CRÉATION DES FEATURES UTILISATEUR :")
df_customers = create_age_bins(df_customers)
print("  ✓ Tranches d'âge créées")
print("\nDistribution des tranches d'âge :")
print(df_customers['age_bin'].value_counts().sort_index())


# 1. Filtrer les utilisateurs et articles actifs
print("\n🔍 FILTRAGE DES UTILISATEURS ET ARTICLES ACTIFS :")
user_interactions = df_transactions.groupby('customer_id').size()
item_interactions = df_transactions.groupby('article_id').size()

active_users = user_interactions[user_interactions >= 5].index
active_items = item_interactions[item_interactions >= 10].index

print(f"  ✓ Utilisateurs actifs (≥5 interactions) : {len(active_users):,} / {len(user_interactions):,} ({len(active_users)/len(user_interactions)*100:.1f}%)")
print(f"  ✓ Articles actifs (≥10 interactions)   : {len(active_items):,} / {len(item_interactions):,} ({len(active_items)/len(item_interactions)*100:.1f}%)")

sampled_transactions = df_transactions[
    df_transactions['customer_id'].isin(active_users) &
    df_transactions['article_id'].isin(active_items)
]
print(f"  ✓ Interactions filtrées : {len(sampled_transactions):,} / {len(df_transactions):,} ({len(sampled_transactions)/len(df_transactions)*100:.1f}%)")


🔧 CRÉATION DES FEATURES UTILISATEUR :
  ✓ Tranches d'âge créées

Distribution des tranches d'âge :
age_bin
0-18         9553
19-25      347616
26-35      393292
36-45      169041
46-55      253951
56-65      139064
65+         43602
Inconnu     15861
Name: count, dtype: int64

🔍 FILTRAGE DES UTILISATEURS ET ARTICLES ACTIFS :
  ✓ Utilisateurs actifs (≥5 interactions) : 925,558 / 1,362,281 (67.9%)
  ✓ Articles actifs (≥10 interactions)   : 83,319 / 104,547 (79.7%)
  ✓ Interactions filtrées : 30,702,763 / 31,788,324 (96.6%)


In [54]:
len(sampled_transactions)

30702763

In [57]:


# 2. Créer des échantillons de différentes tailles
print(f"\n🎲 CRÉATION DES ÉCHANTILLONS :")
#user_sample_sizes = [1000, 10000, 50000]
user_sample_sizes = [10000]
samples = {}
all_active_users_in_sample = sampled_transactions['customer_id'].unique()
all_active_items_in_sample = sampled_transactions['article_id'].unique()

print(f"  • All transactions : {len(sampled_transactions):,}")
print(f"  • Utilisateurs actifs : {len(all_active_users_in_sample):,}")
print(f"  • Articles actifs : {len(all_active_items_in_sample):,}")

for size in user_sample_sizes:
    np.random.seed(42)
    actual_size = min(size, len(all_active_users_in_sample))
    sample_user_ids = np.random.choice(all_active_users_in_sample, size=actual_size, replace=False)
    samples[size] = sampled_transactions[sampled_transactions['customer_id'].isin(sample_user_ids)]

    user_int = samples[size].groupby('customer_id').size()
    item_int = samples[size].groupby('article_id').size()

    print(f"\n  📦 Échantillon {size:,} utilisateurs :")
    print(f"    ├─ Utilisateurs : {samples[size]['customer_id'].nunique():,}")
    print(f"    ├─ Articles : {samples[size]['article_id'].nunique():,}")
    print(f"    ├─ Interactions : {len(samples[size]):,}")
    print(f"    ├─ Moy. interactions/utilisateur : {user_int.mean():.2f}")
    print(f"    └─ Moy. interactions/article : {item_int.mean():.2f}")

sample_sizes = user_sample_sizes
print(f"\n✅ {len(sample_sizes)} échantillons créés et prêts pour l'entraînement")

data = samples[10000].copy()


🎲 CRÉATION DES ÉCHANTILLONS :
  • All transactions : 30,702,763
  • Utilisateurs actifs : 925,556
  • Articles actifs : 83,319

  📦 Échantillon 10,000 utilisateurs :
    ├─ Utilisateurs : 10,000
    ├─ Articles : 51,752
    ├─ Interactions : 333,779
    ├─ Moy. interactions/utilisateur : 33.38
    └─ Moy. interactions/article : 6.45

✅ 1 échantillons créés et prêts pour l'entraînement


In [60]:
# ────────────────────────────────────────────────────────────────
# ÉTAPE 3 : PRÉTRAITEMENT & CONSTRUCTION DATASET LIGHTFM
# ────────────────────────────────────────────────────────────────
print("\n[ÉTAPE 3] Prétraitement & Construction Dataset LightFM")

unique_users = data['customer_id'].unique()
unique_items = data['article_id'].unique()

# Préparation des features d'articles
print("  ► Préparation des features d'articles...")
sample_articles_df = df_articles[df_articles['article_id'].isin(unique_items)].copy()
# (Utilisation de votre liste complète de features)
item_feature_cols = [
    'product_type_name', 
    'product_group_name', 
    'department_name', 
    'colour_group_name', 
    'graphical_appearance_name',
    'perceived_colour_value_name',
    'perceived_colour_master_name',
    'index_group_name',
    'section_name',
    'garment_group_name'   
]

for col in item_feature_cols:
    sample_articles_df[col] = sample_articles_df[col].fillna('Inconnu')

item_features_list = []
for _, row in sample_articles_df.iterrows():
    features = [f"{col}:{row[col]}" for col in item_feature_cols]
    item_features_list.append((row['article_id'], features))
all_item_features = [f for _, features in item_features_list for f in features]

# Préparation des features utilisateur
print("  ► Préparation des features utilisateur...")
sample_users_df = data[['customer_id']].drop_duplicates()
sample_users_df = sample_users_df.merge(
    df_customers[['customer_id', 'age_bin']],
    on='customer_id',
    how='left'
)
if 'Inconnu' not in sample_users_df['age_bin'].cat.categories:
    sample_users_df['age_bin'] = sample_users_df['age_bin'].cat.add_categories('Inconnu')
sample_users_df['age_bin'] = sample_users_df['age_bin'].fillna('Inconnu')

user_feature_cols = ['age_bin']
user_features_list = []
for _, row in sample_users_df.iterrows():
    features = [f"{col}:{row[col]}" for col in user_feature_cols]
    user_features_list.append((row['customer_id'], features))
all_user_features = [f for _, features in user_features_list for f in features]

# Initialisation et Fit du Dataset
print("  ► Initialisation et fit du Dataset LightFM (avec features)...")
dataset = Dataset()
dataset.fit(
    users=unique_users,
    items=unique_items,
    user_features=all_user_features,
    item_features=all_item_features
)

user_id_map, _, item_id_map, _ = dataset.mapping()
user_id_reverse = {v: k for k, v in user_id_map.items()}
item_id_reverse = {v: k for k, v in item_id_map.items()}

# ────────────────────────────────────────────────────────────────
# ÉTAPE 4 : DIVISION TRAIN/TEST TEMPORELLE
# ────────────────────────────────────────────────────────────────
print("\n[ÉTAPE 4] Division Train/Test (split temporel 80/20)")

data_sorted = data.sort_values('t_dat')
split_point = int(0.8 * len(data_sorted))

cutoff_date = data_sorted.iloc[split_point]['t_dat']
print(f"\n📅 Date de cutoff: {cutoff_date}")

train_data = data_sorted.iloc[:split_point]
test_data = data_sorted.iloc[split_point:]

# Binarisation et disjonction
train_data = train_data.drop_duplicates(subset=['customer_id', 'article_id'])
test_data = test_data.drop_duplicates(subset=['customer_id', 'article_id'])
train_pairs_set = set(zip(train_data['customer_id'], train_data['article_id']))
test_pairs_list = list(zip(test_data['customer_id'], test_data['article_id']))
mask = [pair not in train_pairs_set for pair in test_pairs_list]
test_data = test_data[mask] # test_data est maintenant le set 'Warm-Start'

# Construction des matrices 'Warm-Start'
print("  ► Construction des matrices train/test (Warm-Start) via Dataset...")
train_matrix = build_interaction_matrix_with_dataset(dataset, train_data)
test_matrix = build_interaction_matrix_with_dataset(dataset, test_data)

print(f"  ✓ Train (Warm) : {train_matrix.nnz:,} interactions")
print(f"  ✓ Test (Warm)  : {test_matrix.nnz:,} interactions")

print("  ► Construction des matrices de features...")
item_features_matrix = dataset.build_item_features(item_features_list, normalize=True)
user_features_matrix = dataset.build_user_features(user_features_list, normalize=True)


[ÉTAPE 3] Prétraitement & Construction Dataset LightFM
  ► Préparation des features d'articles...
  ► Préparation des features utilisateur...
  ► Initialisation et fit du Dataset LightFM (avec features)...

[ÉTAPE 4] Division Train/Test (split temporel 80/20)

📅 Date de cutoff: 2020-05-04
  ► Construction des matrices train/test (Warm-Start) via Dataset...
  ✓ Train (Warm) : 229,277 interactions
  ✓ Test (Warm)  : 57,752 interactions
  ► Construction des matrices de features...


In [62]:
print(f"\n 📊 Matrices d'interactions (Step 4) :")
print(f"   Train : {train_matrix.shape} - {train_matrix.nnz:,} interactions")
print(f"   Test  : {test_matrix.shape} - {test_matrix.nnz:,} interactions")

print(f"\n 📊 Matrice de features :")
print(f"   Shape : {item_features_matrix.shape}")
print(f"   NNZ   : {item_features_matrix.nnz:,}")


print(f"   ✓ item_features_matrix.npz : {item_features_matrix.shape}")
print(f"   ✓ user_features_matrix.npz : {user_features_matrix.shape}")
print(f"   ✓ dataset.pkl")


 📊 Matrices d'interactions (Step 4) :
   Train : (10000, 51752) - 229,277 interactions
   Test  : (10000, 51752) - 57,752 interactions

 📊 Matrice de features :
   Shape : (51752, 52315)
   NNZ   : 569,272
   ✓ item_features_matrix.npz : (51752, 52315)
   ✓ user_features_matrix.npz : (10000, 10008)
   ✓ dataset.pkl


In [63]:
print("=" * 80)
print("ENTRAÎNEMENT MODÈLE HYBRIDE AVEC ÉVALUATION PAR EPOCH")
print("=" * 80)

K_EVAL = 10

# Créer le modèle hybride
hybrid_model = LightFM(
    loss='warp',
    no_components=30,
    learning_rate=0.1,
    item_alpha=0.01,
    user_alpha=0.01,
    random_state=42
)

print(f"\n🔄 Entraînement en cours avec évaluation à chaque epoch...")
print(f"   Avec item_features + user_features")

import time
start_time = time.time()

# Paramètres d'entraînement
n_epochs = 10

# Stocker les métriques par epoch
metrics_history = {
    'epoch': [],
    'train_auc': [],
    'test_auc': [],
    'test_precision': [],
    'test_recall': [],
    'epoch_time': []
}

print(f"\n{'Epoch':<6} | {'Train AUC':<10} | {'Test AUC':<10} | {'Test P@{K_EVAL}':<12} | {'Test R@{K_EVAL}':<12} | {'Time':<8}")
print("-" * 75)

# Entraînement itératif avec évaluation
for epoch in range(n_epochs):
    epoch_start = time.time()
    
    # Entraîner 1 epoch
    hybrid_model.fit_partial(
        interactions=train_matrix,
        item_features=item_features_matrix,
        user_features=user_features_matrix,
        epochs=1,
        num_threads=4,
        verbose=False
    )
    
    if (epoch + 1) % 2 == 0:
        # Évaluer sur train (AUC uniquement, rapide)
        train_auc = auc_score(
            hybrid_model,
            train_matrix,
            item_features=item_features_matrix,
            user_features=user_features_matrix,
            num_threads=4
        ).mean()
        
        # Évaluer sur test (toutes métriques)
        test_auc = auc_score(
            hybrid_model,
            test_matrix,
            train_interactions=train_matrix,
            item_features=item_features_matrix,
            user_features=user_features_matrix,
            num_threads=4
        ).mean()
        
        test_precision = precision_at_k(
            hybrid_model,
            test_matrix,
            k=K_EVAL,
            train_interactions=train_matrix,
            item_features=item_features_matrix,
            user_features=user_features_matrix,
            num_threads=4
        ).mean()
        
        test_recall = recall_at_k(
            hybrid_model,
            test_matrix,
            k=K_EVAL,
            train_interactions=train_matrix,
            item_features=item_features_matrix,
            user_features=user_features_matrix,
            num_threads=4
        ).mean()
        
        epoch_time = time.time() - epoch_start
        
        # Stocker les métriques
        metrics_history['epoch'].append(epoch + 1)
        metrics_history['train_auc'].append(train_auc)
        metrics_history['test_auc'].append(test_auc)
        metrics_history['test_precision'].append(test_precision)
        metrics_history['test_recall'].append(test_recall)
        metrics_history['epoch_time'].append(epoch_time)
        
        # Afficher les résultats
        print(f"{epoch:<6} | {train_auc:<10.4f} | {test_auc:<10.4f} | "
            f"{test_precision:<12.4f} | {test_recall:<12.4f} | {epoch_time:<8.2f}s")

training_time = time.time() - start_time
print(f"\n ✅ Entraînement terminé en {training_time:.1f}s")

# Résumé des meilleures métriques
print(f"\n📊 RÉSUMÉ:")
best_test_auc_idx = np.argmax(metrics_history['test_auc'])
best_test_prec_idx = np.argmax(metrics_history['test_precision'])

print(f"   • Meilleur Test AUC: {metrics_history['test_auc'][best_test_auc_idx]:.4f} (epoch {best_test_auc_idx + 1})")
print(f"   • Meilleur Test P@{K_EVAL}: {metrics_history['test_precision'][best_test_prec_idx]:.4f} (epoch {best_test_prec_idx + 1})")
print(f"   • Temps moyen/epoch: {np.mean(metrics_history['epoch_time']):.2f}s")




ENTRAÎNEMENT MODÈLE HYBRIDE AVEC ÉVALUATION PAR EPOCH

🔄 Entraînement en cours avec évaluation à chaque epoch...
   Avec item_features + user_features

Epoch  | Train AUC  | Test AUC   | Test P@{K_EVAL} | Test R@{K_EVAL} | Time    
---------------------------------------------------------------------------
1      | 0.6785     | 0.6803     | 0.0005       | 0.0006       | 125.24  s
3      | 0.6855     | 0.6846     | 0.0018       | 0.0020       | 125.01  s
5      | 0.6871     | 0.6855     | 0.0027       | 0.0029       | 125.27  s
7      | 0.6876     | 0.6860     | 0.0029       | 0.0031       | 125.35  s
9      | 0.6875     | 0.6862     | 0.0032       | 0.0035       | 124.13  s

 ✅ Entraînement terminé en 627.8s

📊 RÉSUMÉ:
   • Meilleur Test AUC: 0.6862 (epoch 5)
   • Meilleur Test P@10: 0.0032 (epoch 5)
   • Temps moyen/epoch: 125.00s


In [64]:
start_time = time.time()
# Créer le modèle
hybrid_model = LightFM(
    loss='warp',
    no_components=30,
    learning_rate=0.1,
    item_alpha=0.01,
    user_alpha=0.01,
    random_state=42
)

print("-"*70)
print("🚀 Début de l’entraînement du modèle hybride...")

print("="*70)
print("📊 Évaluation du modèle hybride...")
print("="*70)


# Entraîner
n_epochs = 10
hybrid_model.fit(
    interactions=train_matrix,
    item_features=item_features_matrix,
    user_features=user_features_matrix,
    epochs=n_epochs,
    num_threads=4,
    verbose=True
)

# Évaluer
k=5
print(f"📈 Résultats du modèle hybride (k={k}):")

train_prec = precision_at_k(hybrid_model, train_matrix, k=K_EVAL, item_features=item_features_matrix, user_features=user_features_matrix, num_threads=4).mean()
test_prec = precision_at_k(hybrid_model, test_matrix, k=K_EVAL, train_interactions=train_matrix, item_features=item_features_matrix, user_features=user_features_matrix, num_threads=4).mean()
test_recall = recall_at_k(hybrid_model, test_matrix, k=K_EVAL, train_interactions=train_matrix, item_features=item_features_matrix, user_features=user_features_matrix, num_threads=4).mean()
test_auc = auc_score(hybrid_model, test_matrix, train_interactions=train_matrix, item_features=item_features_matrix, user_features=user_features_matrix, num_threads=4).mean()
training_time = time.time() - start_time


print(f"   - Precision@K (train): {train_prec:.4f}")
print(f"   - Precision@K (test):  {test_prec:.4f}")
print(f"   - Recall@K (test):     {test_recall:.4f}")
print(f"   - AUC (test):          {test_auc:.4f}")
print("-"*70)
print("🏁 Fin de l'évaluation du modèle hybride.")
print("="*70)

----------------------------------------------------------------------
🚀 Début de l’entraînement du modèle hybride...
📊 Évaluation du modèle hybride...
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
📈 Résultats du modèle hybride (k=5):
   - Precision@K (train): 0.0121
   - Precision@K (test):  0.0032
   - Recall@K (test):     0.0035
   - AUC (test):          0.6862
----------------------------------------------------------------------
🏁 Fin de l'évaluation du modèle hybride.


In [65]:
start_time = time.time()
# Créer le modèle
cf_pure_model = LightFM(
    loss='warp',
    no_components=30,
    learning_rate=0.1,
    item_alpha=0.01,
    user_alpha=0.01,
    random_state=42
)

print("-"*70)
print("🚀 Début de l’entraînement du modèle hybride...")

print("="*70)
print("📊 Évaluation du modèle hybride...")
print("="*70)


# Entraîner
n_epochs = 10
cf_pure_model.fit(
    interactions=train_matrix,
    epochs=n_epochs,
    num_threads=4,
    verbose=True
)
# Évaluer
k=5
print(f"📈 Résultats du modèle hybride (k={k}):")

train_prec = precision_at_k(cf_pure_model, train_matrix, k=k, num_threads=4).mean()
test_prec = precision_at_k(cf_pure_model, test_matrix, k=k, train_interactions=train_matrix, num_threads=4).mean()
test_recall = recall_at_k(cf_pure_model, test_matrix, k=k, train_interactions=train_matrix, num_threads=4).mean()
test_auc = auc_score(cf_pure_model, test_matrix, train_interactions=train_matrix, num_threads=4).mean()
training_time = time.time() - start_time


print(f"   - Precision@K (train): {train_prec:.4f}")
print(f"   - Precision@K (test):  {test_prec:.4f}")
print(f"   - Recall@K (test):     {test_recall:.4f}")
print(f"   - AUC (test):          {test_auc:.4f}")
print("-"*70)
print("🏁 Fin de l'évaluation du modèle hybride.")
print("="*70)

----------------------------------------------------------------------
🚀 Début de l’entraînement du modèle hybride...
📊 Évaluation du modèle hybride...
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
📈 Résultats du modèle hybride (k=5):
   - Precision@K (train): 0.0006
   - Precision@K (test):  0.0002
   - Recall@K (test):     0.0001
   - AUC (test):          0.5025
----------------------------------------------------------------------
🏁 Fin de l'évaluation du modèle hybride.


In [66]:
K_VALUES = [5, 10, 20, 50]

print("=" * 80)
print("COMPARAISON CF PUR vs HYBRID MODEL")
print("=" * 80)
print(f"\n 🔄 Évaluation des deux modèles (K={K_VALUES})...")
results_comparison = {
    'cf_pure': {},
    'hybrid': {}
}

# Évaluer CF pur (sur les matrices Step 4)
print(f"\n1️⃣  CF PUR (Step 6):")
print(f"{'K':<6} | {'Precision@K':<13} | {'Recall@K':<13} | {'AUC':<10}")
print("-" * 55)
for k in K_VALUES:
    prec = precision_at_k(cf_pure_model, test_matrix, k=k,
                         train_interactions=train_matrix, num_threads=4).mean()
    rec = recall_at_k(cf_pure_model, test_matrix, k=k,
                     train_interactions=train_matrix, num_threads=4).mean()
    auc = auc_score(cf_pure_model, test_matrix,
                   train_interactions=train_matrix, num_threads=4).mean()
    results_comparison['cf_pure'][k] = {
        'precision': prec,
        'recall': rec,
        'auc': auc
    }
    print(f"{k:<6} | {prec:<13.4f} | {rec:<13.4f} | {auc:<10.4f}")

# Évaluer Hybrid (sur les matrices Step 4 + Features LightFM)
print(f"\n2️⃣  HYBRID MODEL (avec item features LightFM):")
print(f"{'K':<6} | {'Precision@K':<13} | {'Recall@K':<13} | {'AUC':<10}")
print("-" * 55)
for k in K_VALUES:
    # DIFFÉRENCE CLÉ : Utiliser test_interactions (Step 4) et item_features_matrix (LightFM)
    prec = precision_at_k(hybrid_model, test_matrix, k=k,
                         train_interactions=train_matrix,
                         item_features=item_features_matrix, 
                         user_features=user_features_matrix,
                         num_threads=4).mean()
    rec = recall_at_k(hybrid_model, test_matrix, k=k,
                     train_interactions=train_matrix,
                     item_features=item_features_matrix, 
                     user_features=user_features_matrix,
                     num_threads=4).mean()
    auc = auc_score(hybrid_model, test_matrix,
                   train_interactions=train_matrix,
                   item_features=item_features_matrix, 
                   user_features=user_features_matrix,
                   num_threads=4).mean()
    results_comparison['hybrid'][k] = {
        'precision': prec,
        'recall': rec,
        'auc': auc
    }
    print(f"{k:<6} | {prec:<13.4f} | {rec:<13.4f} | {auc:<10.4f}")
# Calculer l'amélioration
print(f"\n{'='*80}")
print("📊 AMÉLIORATION HYBRID vs CF PUR")
print(f"{'='*80}")
print(f"\n{'K':<6} | {'ΔPrecision@K':<15} | {'ΔRecall@K':<15} | {'ΔAUC':<10}")
print("-" * 60)
for k in K_VALUES:
    delta_prec = results_comparison['hybrid'][k]['precision'] - results_comparison['cf_pure'][k]['precision']
    delta_rec = results_comparison['hybrid'][k]['recall'] - results_comparison['cf_pure'][k]['recall']
    delta_auc = results_comparison['hybrid'][k]['auc'] - results_comparison['cf_pure'][k]['auc']
    print(f"{k:<6} | {delta_prec:>+14.4f} | {delta_rec:>+14.4f} | {delta_auc:>+9.4f}")
print(f"\n✅ Comparaison terminée")


COMPARAISON CF PUR vs HYBRID MODEL

 🔄 Évaluation des deux modèles (K=[5, 10, 20, 50])...

1️⃣  CF PUR (Step 6):
K      | Precision@K   | Recall@K      | AUC       
-------------------------------------------------------
5      | 0.0002        | 0.0001        | 0.5025    
10     | 0.0003        | 0.0002        | 0.5025    
20     | 0.0002        | 0.0003        | 0.5025    
50     | 0.0002        | 0.0009        | 0.5025    

2️⃣  HYBRID MODEL (avec item features LightFM):
K      | Precision@K   | Recall@K      | AUC       
-------------------------------------------------------
5      | 0.0027        | 0.0015        | 0.6862    
10     | 0.0032        | 0.0035        | 0.6862    
20     | 0.0018        | 0.0042        | 0.6862    
50     | 0.0015        | 0.0088        | 0.6862    

📊 AMÉLIORATION HYBRID vs CF PUR

K      | ΔPrecision@K    | ΔRecall@K       | ΔAUC      
------------------------------------------------------------
5      |        +0.0025 |        +0.0014 |   +0.1837
10

In [67]:
from scipy.sparse import csr_matrix

# --- sécurité : tout en CSR pour le slicing
train_matrix = train_matrix.tocsr()
test_matrix  = test_matrix.tocsr()
user_features_matrix = user_features_matrix.tocsr()
item_features_matrix = item_features_matrix.tocsr()

def keep_only_cols(M, cols):
    M = M.tocsr()
    mask = np.zeros(M.shape[1], dtype=bool)
    mask[cols] = True
    return M[:, mask]

def keep_only_rows(M, rows):
    M = M.tocsr()
    return M[rows, :]

# ------- A. Cold-start ITEMS -------
# I_cold = np.where(train_item_deg == 0)[0]   # déjà calculé chez toi
test_cold_items  = keep_only_cols(test_matrix,  I_cold)
train_cold_items = keep_only_cols(train_matrix, I_cold)  # <-- aligner train !
item_feat_cold   = keep_only_rows(item_features_matrix, I_cold)  # <-- mêmes items

# users complets côté items-cold
user_feat_full   = user_features_matrix

# ------- B. Cold-start USERS -------
# U_cold = np.where(train_user_deg == 0)[0]
test_cold_users  = keep_only_rows(test_matrix,  U_cold)
train_cold_users = keep_only_rows(train_matrix, U_cold)  # <-- aligner train !
user_feat_cold   = keep_only_rows(user_features_matrix, U_cold)  # <-- mêmes users

# items complets côté users-cold
item_feat_full   = item_features_matrix

from lightfm.evaluation import precision_at_k, recall_at_k, auc_score

def eval_segment(model, test_seg, train_seg, user_feat, item_feat, k=10):
    prec = precision_at_k(
        model, test_seg, k=k,
        train_interactions=train_seg,            # <-- shape identique à test_seg
        user_features=user_feat,
        item_features=item_feat,
        num_threads=4
    ).mean()
    rec = recall_at_k(
        model, test_seg, k=k,
        train_interactions=train_seg,
        user_features=user_feat,
        item_features=item_feat,
        num_threads=4
    ).mean()
    auc = auc_score(
        model, test_seg,
        train_interactions=train_seg,
        user_features=user_feat,
        item_features=item_feat,
        num_threads=4
    ).mean()
    return prec, rec, auc

for K in [5,10,20,50]:
    # ITEMS-COLD: mêmes users, sous-ensemble d'items
    p_i, r_i, a_i = eval_segment(
        hybrid_model,
        test_cold_items, train_cold_items,
        user_feat=user_feat_full,    # mêmes users
        item_feat=item_feat_cold,    # items filtrés
        k=K
    )
    # USERS-COLD: mêmes items, sous-ensemble d'users
    p_u, r_u, a_u = eval_segment(
        hybrid_model,
        test_cold_users, train_cold_users,
        user_feat=user_feat_cold,    # users filtrés
        item_feat=item_feat_full,    # mêmes items
        k=K
    )
    print(f"K={K} | ITEMS-COLD -> P@K={p_i:.4f} R@K={r_i:.4f} AUC={a_i:.4f}")
    print(f"      USERS-COLD -> P@K={p_u:.4f} R@K={r_u:.4f} AUC={a_u:.4f}")

K=5 | ITEMS-COLD -> P@K=0.0015 R@K=0.0015 AUC=0.6363
      USERS-COLD -> P@K=0.0044 R@K=0.0024 AUC=0.6872
K=10 | ITEMS-COLD -> P@K=0.0029 R@K=0.0065 AUC=0.6363
      USERS-COLD -> P@K=0.0038 R@K=0.0038 AUC=0.6872
K=20 | ITEMS-COLD -> P@K=0.0021 R@K=0.0097 AUC=0.6363
      USERS-COLD -> P@K=0.0024 R@K=0.0046 AUC=0.6872
K=50 | ITEMS-COLD -> P@K=0.0018 R@K=0.0179 AUC=0.6363
      USERS-COLD -> P@K=0.0021 R@K=0.0103 AUC=0.6872


In [70]:
import numpy as np
from scipy.sparse import csr_matrix, diags
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score

# --- Sécu formats
train_matrix = train_matrix.tocsr()
test_matrix  = test_matrix.tocsr()

# --- Degrés (pour détecter le cold-start)
train_user_deg = np.asarray(train_matrix.getnnz(axis=1)).ravel()
train_item_deg = np.asarray(train_matrix.getnnz(axis=0)).ravel()

U_cold = np.where(train_user_deg == 0)[0]   # users jamais vus en train
I_cold = np.where(train_item_deg == 0)[0]   # items jamais vus en train

n_users, n_items = train_matrix.shape

# ===== Helpers: masques qui CONSERVERNT la shape =====
def mask_cols_keep_shape(M, cols_keep_idx):
    """Garde la même shape, met à 0 les colonnes NON sélectionnées."""
    mask = np.zeros(n_items, dtype=np.float32)
    mask[cols_keep_idx] = 1.0
    D = diags(mask, offsets=0, shape=(n_items, n_items), format="csr")
    return (M @ D).tocsr()

def mask_rows_keep_shape(M, rows_keep_idx):
    """Garde la même shape, met à 0 les lignes NON sélectionnées."""
    mask = np.zeros(n_users, dtype=np.float32)
    mask[rows_keep_idx] = 1.0
    D = diags(mask, offsets=0, shape=(n_users, n_users), format="csr")
    return (D @ M).tocsr()

# ===== Segments cold en CONSERVANT les shapes =====
# ITEMS-COLD: mêmes users, seules les colonnes I_cold sont actives
test_items_cold  = mask_cols_keep_shape(test_matrix,  I_cold)
train_items_cold = mask_cols_keep_shape(train_matrix, I_cold)  # devrait être tout zéro par définition

# USERS-COLD: mêmes items, seules les lignes U_cold sont actives
test_users_cold  = mask_rows_keep_shape(test_matrix,  U_cold)
train_users_cold = mask_rows_keep_shape(train_matrix, U_cold)  # devrait être tout zéro par définition

# Sanity checks (shapes inchangées)
assert test_items_cold.shape  == train_matrix.shape
assert train_items_cold.shape == train_matrix.shape
assert test_users_cold.shape  == train_matrix.shape
assert train_users_cold.shape == train_matrix.shape

# ===== Évaluations CF (sans features) =====
def eval_cf(model, test_seg, train_seg, k):
    prec = precision_at_k(model, test_seg, k=k, train_interactions=train_seg, num_threads=4).mean()
    rec  = recall_at_k(   model, test_seg, k=k, train_interactions=train_seg, num_threads=4).mean()
    auc  = auc_score(     model, test_seg,      train_interactions=train_seg, num_threads=4).mean()
    return prec, rec, auc

K_VALUES = [5,10,20,50]

print("\n=== CF • COLD-START ITEMS ===")
print(f"test nnz={test_items_cold.nnz}, train nnz={train_items_cold.nnz} (devrait être 0)")
for k in K_VALUES:
    p, r, a = eval_cf(cf_pure_model, test_items_cold, train_items_cold, k)
    print(f"K={k:<3} | P@K={p:.4f}  R@K={r:.4f}  AUC={a:.4f}")

print("\n=== CF • COLD-START USERS ===")
print(f"test nnz={test_users_cold.nnz}, train nnz={train_users_cold.nnz} (devrait être 0)")
for k in K_VALUES:
    p, r, a = eval_cf(cf_pure_model, test_users_cold, train_users_cold, k)
    print(f"K={k:<3} | P@K={p:.4f}  R@K={r:.4f}  AUC={a:.4f}")


=== CF • COLD-START ITEMS ===
test nnz=24840, train nnz=0 (devrait être 0)
K=5   | P@K=0.0002  R@K=0.0002  AUC=0.4470
K=10  | P@K=0.0002  R@K=0.0003  AUC=0.4470
K=20  | P@K=0.0001  R@K=0.0005  AUC=0.4470
K=50  | P@K=0.0001  R@K=0.0009  AUC=0.4470

=== CF • COLD-START USERS ===
test nnz=4149, train nnz=0 (devrait être 0)
K=5   | P@K=0.0004  R@K=0.0003  AUC=0.5014
K=10  | P@K=0.0006  R@K=0.0011  AUC=0.5014
K=20  | P@K=0.0005  R@K=0.0014  AUC=0.5014
K=50  | P@K=0.0003  R@K=0.0021  AUC=0.5014


In [71]:
import numpy as np
from scipy.sparse import csr_matrix, diags
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score

# ===============================================================
#   COLD-START TEST — COMPARAISON CF PUR vs HYBRID LightFM
# ===============================================================

print("="*80)
print("COMPARAISON CF PUR vs HYBRID MODEL — COLD START")
print("="*80)
print("\n 🔄 Évaluation des deux modèles (K=[5, 10, 20, 50])...\n")

# --- Sécurité formats
train_matrix = train_matrix.tocsr()
test_matrix  = test_matrix.tocsr()

user_features_matrix = user_features_matrix.tocsr()
item_features_matrix = item_features_matrix.tocsr()

n_users, n_items = train_matrix.shape

# --- Détection des cold-start users
train_user_deg = np.asarray(train_matrix.getnnz(axis=1)).ravel()
U_cold = np.where(train_user_deg == 0)[0]
print(f"👥 Users cold : {len(U_cold)} / {n_users}")


def mask_rows_keep_shape(M, rows_keep_idx):
    mask = np.zeros(n_users, dtype=np.float32)
    mask[rows_keep_idx] = 1.0
    D = diags(mask, offsets=0, shape=(n_users, n_users), format="csr")
    return (D @ M).tocsr()

# --- Segments cold (shapes inchangées)
test_users_cold  = mask_rows_keep_shape(test_matrix,  U_cold)
train_users_cold = mask_rows_keep_shape(train_matrix, U_cold)

# --- Fonctions d’évaluation
def eval_cf(model, test_seg, train_seg, k):
    prec = precision_at_k(model, test_seg, k=k, train_interactions=train_seg, num_threads=4).mean()
    rec  = recall_at_k(   model, test_seg, k=k, train_interactions=train_seg, num_threads=4).mean()
    auc  = auc_score(     model, test_seg,      train_interactions=train_seg, num_threads=4).mean()
    return prec, rec, auc

def eval_hybrid(model, test_seg, train_seg, k, user_feat, item_feat):
    prec = precision_at_k(model, test_seg, k=k, train_interactions=train_seg,
                          user_features=user_feat, item_features=item_feat, num_threads=4).mean()
    rec  = recall_at_k(   model, test_seg, k=k, train_interactions=train_seg,
                          user_features=user_feat, item_features=item_feat, num_threads=4).mean()
    auc  = auc_score(     model, test_seg,      train_interactions=train_seg,
                          user_features=user_feat, item_features=item_feat, num_threads=4).mean()
    return prec, rec, auc



# ===============================================================
#   ÉVALUATION — USERS COLD
# ===============================================================

cf_users = {}
hyb_users = {}

print("\n2️⃣  USERS-COLD (utilisateurs jamais vus pendant l'entraînement):\n")

for k in K_VALUES:
    p_cf, r_cf, a_cf = eval_cf(cf_pure_model, test_users_cold, train_users_cold, k)
    p_hy, r_hy, a_hy = eval_hybrid(hybrid_model, test_users_cold, train_users_cold, k,
                                   user_features_matrix, item_features_matrix)
    cf_users[k] = {'p': p_cf, 'r': r_cf, 'a': a_cf}
    hyb_users[k] = {'p': p_hy, 'r': r_hy, 'a': a_hy}

print("CF PUR (Step 6):")
print("K      | Precision@K   | Recall@K      | AUC       ")
print("-------------------------------------------------------")
for k in K_VALUES:
    v = cf_users[k]
    print(f"{k:<6} | {v['p']:<13.4f} | {v['r']:<13.4f} | {v['a']:<.4f}")
print()

print("HYBRID MODEL (avec user features LightFM):")
print("K      | Precision@K   | Recall@K      | AUC       ")
print("-------------------------------------------------------")
for k in K_VALUES:
    v = hyb_users[k]
    print(f"{k:<6} | {v['p']:<13.4f} | {v['r']:<13.4f} | {v['a']:<.4f}")
print()

print("="*80)
print("📊 AMÉLIORATION HYBRID vs CF PUR — USERS-COLD")
print("="*80)
print("K      | ΔPrecision@K    | ΔRecall@K       | ΔAUC      ")
print("------------------------------------------------------------")
for k in K_VALUES:
    dprec = hyb_users[k]['p'] - cf_users[k]['p']
    drec  = hyb_users[k]['r'] - cf_users[k]['r']
    dauc  = hyb_users[k]['a'] - cf_users[k]['a']
    print(f"{k:<6} | {dprec:+14.4f} | {drec:+14.4f} | {dauc:+10.4f}")
print()

print("✅ Comparaison Cold-Start terminée.")

COMPARAISON CF PUR vs HYBRID MODEL — COLD START

 🔄 Évaluation des deux modèles (K=[5, 10, 20, 50])...

👥 Users cold : 496 / 10000

2️⃣  USERS-COLD (utilisateurs jamais vus pendant l'entraînement):

CF PUR (Step 6):
K      | Precision@K   | Recall@K      | AUC       
-------------------------------------------------------
5      | 0.0004        | 0.0003        | 0.5014
10     | 0.0006        | 0.0011        | 0.5014
20     | 0.0005        | 0.0014        | 0.5014
50     | 0.0003        | 0.0021        | 0.5014

HYBRID MODEL (avec user features LightFM):
K      | Precision@K   | Recall@K      | AUC       
-------------------------------------------------------
5      | 0.0044        | 0.0024        | 0.6872
10     | 0.0038        | 0.0038        | 0.6872
20     | 0.0024        | 0.0046        | 0.6872
50     | 0.0021        | 0.0103        | 0.6872

📊 AMÉLIORATION HYBRID vs CF PUR — USERS-COLD
K      | ΔPrecision@K    | ΔRecall@K       | ΔAUC      
--------------------------------------